In [2]:
import cv2
import os
import re
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mobilenet = models.mobilenet_v3_large(
    weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
).to(device)

feature_extractor = torch.nn.Sequential(
    mobilenet.features,
    mobilenet.avgpool,
    torch.nn.Flatten()
)
feature_extractor.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

IMAGE_DIR = r"E:\TSG\feature boost\3"
SAVE_DIR = "./PC_GradCAM_Results"

def extract_only_deep_feature(img_path):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    tensor = transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = feature_extractor(tensor).squeeze().cpu().numpy()
    return feat

all_files = os.listdir(IMAGE_DIR)
bmp_files = [f for f in all_files if f.lower().endswith(".bmp")]
def extract_num(filename):
    match = re.search(r'(\d+)', filename)
    return int(match.group(1)) if match else float('inf')
bmp_files_sorted = sorted(bmp_files, key=extract_num)

deep_feat_all = []
for fname in bmp_files_sorted:
    pth = os.path.join(IMAGE_DIR, fname)
    ft = extract_only_deep_feature(pth)
    deep_feat_all.append(ft)
deep_feat_all = np.array(deep_feat_all)

scaler_deep = StandardScaler()
deep_scaled = scaler_deep.fit_transform(deep_feat_all)
pca_image = PCA(n_components=30, random_state=42)
pattern_image_pca_full = pca_image.fit_transform(deep_scaled)

class GradCAM:
    def __init__(self, model_extractor, target_layer):
        self.extractor = model_extractor
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_act)
        target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, mod, inp, out):
        self.activations = out.detach().cpu().numpy()
    def _save_grad(self, mod, grad_in, grad_out):
        self.gradients = grad_out[0].detach().cpu().numpy()

    def generate_cam(self, input_tensor, pc_weight):
        self.extractor.zero_grad()
        deep_feat = self.extractor(input_tensor)
        target_score = torch.sum(deep_feat * pc_weight)
        target_score.backward(retain_graph=False)

        acts = self.activations.squeeze()
        grads = self.gradients.squeeze()
        channel_weights = np.mean(grads, axis=(1,2))

        cam = np.zeros(acts.shape[1:])
        for i,w in enumerate(channel_weights):
            cam += w * acts[i]
        cam = np.maximum(cam,0)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, target_score.item()

def overlay_heatmap(img_bgr, cam, alpha=0.4):
    h,w = img_bgr.shape[:2]
    cam_res = cv2.resize(cam, (w,h))
    heat = cv2.applyColorMap((cam_res*255).astype(np.uint8), cv2.COLORMAP_JET)
    overlay = np.uint8(heat*alpha + img_bgr*(1-alpha))
    return overlay

def single_pc_cam(img_path, pc_idx, pca_model, gradcam):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(img_rgb)
    tensor = transform(pil).unsqueeze(0).to(device)
    tensor.requires_grad_(True)

    pc_w = torch.tensor(pca_model.components_[pc_idx], dtype=torch.float32, device=device)
    cam, score = gradcam.generate_cam(tensor, pc_w)
    overlay = overlay_heatmap(img_bgr, cam)
    return overlay, score

def batch_visual(pc_list, top_n=10):
    os.makedirs(SAVE_DIR, exist_ok=True)
    target_layer = mobilenet.features[-1]
    gradcam = GradCAM(feature_extractor, target_layer)

    for pc_idx in pc_list:
        pc_name = f"PC{pc_idx+1}"
        pc_dir = os.path.join(SAVE_DIR, pc_name)
        os.makedirs(os.path.join(pc_dir, "high_score"), exist_ok=True)
        os.makedirs(os.path.join(pc_dir, "low_score"), exist_ok=True)

        scores = pattern_image_pca_full[:, pc_idx]
        high_idx = np.argsort(scores)[-top_n:][::-1]
        low_idx = np.argsort(scores)[:top_n]

        for rank, idx in enumerate(high_idx):
            fname = bmp_files_sorted[idx]
            img_pth = os.path.join(IMAGE_DIR, fname)
            try:
                ov, s = single_pc_cam(img_pth, pc_idx, pca_image, gradcam)
                save_n = f"rank{rank+1}_score{s:.3f}.jpg"
                cv2.imwrite(os.path.join(pc_dir,"high_score",save_n), ov)
            except Exception:
                pass

        for rank, idx in enumerate(low_idx):
            fname = bmp_files_sorted[idx]
            img_pth = os.path.join(IMAGE_DIR, fname)
            try:
                ov, s = single_pc_cam(img_pth, pc_idx, pca_image, gradcam)
                save_n = f"rank{rank+1}_score{s:.3f}.jpg"
                cv2.imwrite(os.path.join(pc_dir,"low_score",save_n), ov)
            except Exception:
                pass
    print(f"Processing finished. Output directory: {os.path.abspath(SAVE_DIR)}")

if __name__ == "__main__":
    SELECT_PC = [0, 1, 2, 3, 4, 6, 7, 8, 10, 12, 15, 20, 22, 27, 29]
    batch_visual(pc_list=SELECT_PC, top_n=len(bmp_files_sorted))

Processing finished. Output directory: E:\TSG\feature boost\Must\MUST\PC_GradCAM_Results
